# 11 — LangGraph Agent

Build a stateful, graph-based agent with tool execution and memory checkpointing.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

## Define Tools

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search an internal knowledge base for information."""
    kb = {
        "refund": "Refund policy: Full refund within 30 days of purchase. After 30 days, store credit only.",
        "shipping": "Standard shipping: 5-7 business days. Express: 1-2 business days ($15 extra).",
        "hours": "Customer support hours: Mon-Fri 9am-6pm EST. Weekend: 10am-4pm EST.",
        "warranty": "All products come with a 1-year manufacturer warranty covering defects.",
    }
    for key, value in kb.items():
        if key in query.lower():
            return value
    return "No relevant information found."

@tool
def create_ticket(subject: str, priority: str) -> str:
    """Create a support ticket. Priority: low, medium, or high."""
    if priority not in ("low", "medium", "high"):
        return "Invalid priority."
    return f"Ticket #{hash(subject) % 10000} created — subject: '{subject}', priority: {priority}"

## Build the Graph

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def chatbot_node(state: AgentState):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_with_tools = llm.bind_tools([search_knowledge_base, create_ticket])
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def should_use_tools(state: AgentState) -> str:
    last_message = state["messages"][-1]
    return "tools" if last_message.tool_calls else END

graph = StateGraph(AgentState)
graph.add_node("chatbot", chatbot_node)
graph.add_node("tools", ToolNode([search_knowledge_base, create_ticket]))
graph.add_edge(START, "chatbot")
graph.add_conditional_edges("chatbot", should_use_tools, {"tools": "tools", END: END})
graph.add_edge("tools", "chatbot")

agent = graph.compile(checkpointer=MemorySaver())

## Run the Agent

In [ ]:
config = {"configurable": {"thread_id": "demo-session"}}

for user_msg in [
    "What is your refund policy?",
    "I'd like to return something I bought 45 days ago. Can you create a ticket for me?",
    "What are your support hours?",
]:
    print(f"User: {user_msg}")
    response = agent.invoke({"messages": [HumanMessage(content=user_msg)]}, config=config)
    print(f"Agent: {response['messages'][-1].content}\n")